# Descriptive Statistics

## Purpose
This worksheet is an opportunity for you to practice calculating descriptive statistics. This is something you will need to do when reporting your results for our experiments. Some of the questions are purposely vague, because I would like you to think about them, and make a qualified decision about how to answer them.

## Part 1: Measures of central tendency.
The first block of code you can just copy and paste. If you need to, uncomment the ``pip intall`` sections. 

In [15]:
# Just copy and paste this whole section into your notebook and run it.

#Install and import libraries

# Install packages if you need them
#%pip install pandas
#%pip install seaborn
#%pip install scipy

# Import packages
import pandas as pd
import seaborn as sns

# read in the data from the url
url = 'https://raw.githubusercontent.com/ethanweed/ExPsyLing/refs/heads/master/experiments/masked_prime/data/combined_data.csv'
raw = pd.read_csv(url)                                                      # read in the data from the url

# make a list of all the columns we want to keep
cols = ['participant_id', 'sender', 'duration', 'prime', 'target', 'condition', 'soa_condition', 'congruence', 'response', 'correct_response']

# filter the data to keep only the columns we want, and only the rows we want
df = raw[cols]                                                             # make a new dataframe with only the columns in cols
df = df[df['sender'] == 'target']                                          # only keep the "target" rows
df = df[df['condition'] != 'practice']                                     # throw away the practice trials

# Some of the target words were labeled incorrectly in the original data, so we need to fix those.
incorrect_labels = ['LAND', 'TREE', 'BEAR', 'ROSE', 'HILL', 'DIRT', 'BULL', 'DUCK']
for word in incorrect_labels:
    df.loc[df['target'] == word, 'correct_response'] = 'n'                 # fix items with incorrect answer lables

df['correct'] = (df['response'] == df['correct_response']).astype(int)     # make a new column with 0 or 1 for incorrect / correct
df = df[df['correct'] == 1]                                                # keep only the correct responses   
df = df.rename(columns={'soa_condition': 'soa', 'duration': 'rt'})         # rename the soa and duration columns, just to look nicer

In [16]:
# On average, how long does it take participants to respond to the target words?
df.tail()

,participant_id,sender,rt,prime,target,condition,soa,congruence,response,correct_response,correct
15606,69,target,699.2,belt,BELT,Con_Synth,short,congruent,m,m,1
15612,69,target,580.2,daisy,DAISY,Con_Nat,short,congruent,n,n,1
15618,69,target,577.9,peach,SHED,Inc_Nat_Synth,short,incongruent,m,m,1
15624,69,target,661.1,rake,HUSKY,Inc_Synth_Nat,short,incongruent,n,n,1
15630,69,target,709.2,fern,PETAL,Inc_Nat_Nat,short,incongruent,n,n,1


In [13]:
print(len(df['participant_id'].unique()))

19


In [10]:
print(df[df['sender'] == 'target']['rt'].mean())

742.4425990348288


In [13]:
print(df[(df['congruence'] == 'congruent') & (df['sender'] == 'target')]['rt'].mean())
print(df[df['congruence'] == 'congruent']['rt'].mean())

723.748791628387
723.748791628387


In [18]:
print(df[df['congruence'] == 'incongruent']['rt'].mean())

761.7234543676751


## Questions

1. How many participants are in the data?
2. On average, how long does it take participants to respond to the target words?
3. If another participant were to do this experiment, how long would you expect them to take to respond to target words with congruent primes?
4. If another participant were to do this experiment, how long would you expect them to take to respond to target words with incongruent primes?
5. Are you equally confident in your answers to the previous two questions? How could you quantify your confidence in your answers? 


Hints: 
1. These are questions about central tendency and variance.
2. There are different ways to quantify each of these. 
  a) Talk with each other about the different options, and what they mean.
  b) Plot the data, and use this to inform your choices.

Bonus (TOTALLY OPTIONAL, ONLY IF YOU RRALLY WANT TO):
These are response time data, and as we talked about back on September 23rd when we read Balota and Yap (2011), response time measures have a tendency to be skewed, rather than normally distributed. Investigate these data to see if they are indeed skewed, and if they are, try fitting an exgaussian distribution instead of a normal distribution for some more fine-tuned measures of central tendency:

```python
from scipy.stats import exponnorm

# Make a function to get the parameters of the ex-gaussian function for each participant and condition
def get_parameters(data):
    K, loc, scale = exponnorm.fit(data)
    tau = K * scale
    return pd.Series({'mu': loc,
                        'sigma': scale,
                        'tau': tau,
                        'mean': loc + tau})


# Use the function from above to get the parameters for each participant and condition
df_params = (
    df.groupby(['participant_id', 'congruence'])['rt']
    .apply(get_parameters)
    .unstack()
    .reset_index()
)

# Inspect the data
df_params.head()

```
Do these measures give you a better estimate the standard measures of central tendency?

 


## Part 2: Correlation

Copy and paste the code below to load the data from the MEGALEX study [<sup id="fn1-back">1</sup>](#fn1) by Ferrand et al (2017)[<sup id="fn2-back">2</sup>](#fn2). These data include (among other things) word recognition response times, word length data, and frequency data.




In [17]:
url = 'https://raw.githubusercontent.com/ethanweed/datasets-for-teaching/refs/heads/main/MEGALEX.csv'
df = pd.read_csv(url)
df.head()

,item,n_trials,err,rt,sd,rtz,n_used,c_freq_movies,lc_freq_movies,c_freq_books,lc_freq_books,n_letters,n_syllables
0,abaissa,25,0.04,721.833333,106.416191,-0.035017,24,0.00,#NAME?,2.64,0.421603927,7,3
1,abaissait,25,0.08,836.782609,257.778086,0.366517,23,0.02,-1.698970004,2.50,0.397940009,9,3
2,abaissant,26,0.00,773.840000,224.378713,0.254392,25,0.04,-1.397940009,2.77,0.442479769,9,3
3,abaisse,25,0.04,698.826087,246.667913,-0.216788,23,1.30,0.113943352,3.45,0.537819095,7,2
4,abaissé,25,0.00,656.782609,177.024897,-0.434544,23,0.77,-0.113509275,1.55,0.190331698,7,3


Here is what the headers refer to.

| Column Header | Meaning |
|---|---|
| item | The specific word or stimulus being tested |
| n_trials | The total number of trials conducted for that item |
| err | The accuracy rate or error percentage for the item |
| rt | The reaction time or response time for the item |
| sd | The standard deviation of the reaction times |
| rtz | The standardised reaction time |
| n_used | The number of participants whose data was used for the item |
| c_freq_movies | The cumulative frequency of the word in movie subtitles |
| lc_freq_movies | The logarithmic cumulative frequency of the word in movie subtitles |
| c_freq_books | The cumulative frequency of the word in books |
| lc_freq_books | The logarithmic cumulative frequency of the word in books |
| n_letters | The number of letters in the word |
| n_syllables | The number of syllables in the word |

In [20]:
print(df['n_letters'].corr(df['c_freq_books']))

-0.09147595866662069


In [18]:
print(df['n_letters'].corr(df['c_freq_movies']))

-0.10198798396341646


In [19]:
print(df['c_freq_books'].corr(df['c_freq_movies']))

0.8483225240617237


In [20]:
print(df['rt'].corr(df['c_freq_movies']))
print(df['rt'].corr(df['c_freq_books']))

-0.04850543778380267
-0.04637396691497746


In [25]:
print("r:", df['sd'].corr(df['c_freq_books']))
print( "r:", df['sd'].corr(df['c_freq_movies']))

r: -0.03549194497527117
r: -0.03674644583262573


## Questions

1. Is there a relationship between word length and word frequency?
2. Does it make a difference if frequency is measured by looking at books or movies?
3. Does the frequency of the word affect your confidence in your estimate of how long people take to recognize it?

Hint: These are all questions about correlation.

### Footnotes

[<sup id="fn1">1</sup>](#fn1-back) Ferrand, L., Méot, A., Spinelli, E., New, B., Pallier, C., Bonin, P., Dufau, S., Mathôt, S., & Grainger, J. (2017). MEGALEX: A megastudy of visual and auditory word recognition. *Behavior Research Methods*.
  
[<sup id="fn2">2</sup>](#fn2-back) [OpenSesame](https://osdoc.cogsci.nl) fans may notice that Sebastian Mathôt is one of the authors of this study.